DH演習2026 · 教材 第38版 · 2026-r38 · 2026-10-15

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第38版
教材ID: 2026-r38
教材日付: 2026-10-15
活動: place_mapping
環境: Colab / Jupyter
ソース: 0168ecfac6edcdfec02b46504654b08d76a08d2f
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第9回：表の行を地図の点にする

上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

資料：第8回の難波橋の判断表から作った2行、[今昔マップ on the web](https://ktgis.net/kjmapw/kjmapw.html?lat=34.6929&lng=135.5064&zoom=18&dataset=keihansin&age=0&screen=2)の旧版地形図「大阪東北部」（1908年測図）、背景地図は[OpenStreetMap](https://www.openstreetmap.org/copyright)

## 1. 表を読む

N-currentはWikidataにある現在の代表点、N-earlierは1915年に移設される前の橋で、位置を保留した行です。何行が地図の点になるか、予測してから実行してください。

In [ ]:
import io

import folium
import pandas as pd

locations = '''record_id,label,period_label,latitude,longitude,location_status,source_url
N-current,難波橋,現代の代表点,34.692694,135.507139,modern,https://www.wikidata.org/w/index.php?title=Q11659546&oldid=2191803708
N-earlier,難波橋,1915年の移設以前,,,unresolved,https://www.city.osaka.lg.jp/kensetsu/page/0000023795.html
'''
rows = pd.read_csv(io.StringIO(locations))
rows

## 2. 座標のある行を選ぶ

空欄を0にすると、欠けた位置を別の場所に描いてしまいます。`dropna`は緯度・経度が空欄の行を除き、元の`rows`は変えません。

In [ ]:
located = rows.dropna(subset=["latitude", "longitude"])
print("地図に置く行：", len(located), "位置を保留する行：", len(rows) - len(located))

## 3. 地図に点を置く

`folium.Marker`には緯度・経度の順で渡します。`zoom`や`color`を変えても、場所の判断は変わりません。点をクリックすると、元の行の表記と時期が表示されます。

In [ ]:
zoom = 16
color = "darkgreen"

m = folium.Map(location=[34.6927, 135.5071], zoom_start=zoom)
for row in located.itertuples():
    folium.Marker([row.latitude, row.longitude], popup=f"{row.label}／{row.period_label}", icon=folium.Icon(color=color)).add_to(m)
m

地図にないN-earlierの行は、表には残っています。その行に座標を入れるには、どの資料の、どの読みが必要でしょうか。

## 確認問題

下の地図の破線は、1908年の旧版地形図から読んだ移設前の難波橋の経路です。破線の上の一点をクリックして座標を確かめ、`answer`に緯度・経度の順で入れてください。

In [ ]:
route = [(34.6920, 135.5057), (34.6924, 135.5060), (34.6932, 135.5065), (34.6939, 135.5070)]
guide = folium.Map(location=route[1], zoom_start=17)
folium.PolyLine(route, dash_array="8 8", tooltip="旧橋の経路（1908年図からの概略）").add_to(guide)
folium.LatLngPopup().add_to(guide)
guide

In [ ]:
answer = None  # (緯度, 経度)

In [ ]:
#@title 答えを確かめる
import itertools
import math


def metres(a, b):
    return ((b[1] - a[1]) * 111320 * math.cos(math.radians(a[0])), (b[0] - a[0]) * 110574)


def away(point):
    best = math.inf
    for start, end in itertools.pairwise(route):
        sx, sy = metres(start, end)
        px, py = metres(start, point)
        t = max(0.0, min(1.0, (px * sx + py * sy) / (sx * sx + sy * sy)))
        best = min(best, math.hypot(px - t * sx, py - t * sy))
    return best


if answer is None:
    print("結果：不合格（`answer`に緯度と経度を入れてください）")
elif math.hypot(*metres((34.692694, 135.507139), answer)) < 30:
    print("結果：不合格（現在の代表点と同じ位置です。破線をたどってください）")
elif away(answer) > 30:
    print(f"結果：不合格（破線から約{away(answer):.0f}メートル離れています。緯度と経度の順も確かめてください）")
else:
    print("結果：合格")